# FireSight-IR | Module 2 v2 — Feature Engineering

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 2 v2 — Feature normalisation, label repair, class weights  
**Platform:** Google Colab  
**Depends on:** Module 1c v2 outputs — `viirs_fp_srf_v2_*.parquet` and `firesight_patches.h5`

---

## Why Module 2 was re-run

Module 2 (original) was run before Module 1c v2 completed. That version:
- Computed scalers on zero-filled land cover (all `lc_*` columns = 0 from ESA CCI failure)
- Applied label repair to v1 parquets, producing 42,018 false-alarm pixels
- Saved class weights of `[18.98, 1.0, 27.02]` based on that inflated false-alarm count

After Module 1c v2:
- MODIS MCD12Q1 land cover is real (8 columns with genuine variance)
- False-alarm count is 23,706 (more conservative MODIS urban + OSM gate)
- Scalers must be recomputed on real land cover data
- Class weights must reflect the updated label distribution
- The `mlp_srf` HDF5 vectors already updated in-place by Module 1c v2

## What this module does

1. **Feature audit** — verify all feature columns against v2 parquets
2. **Label inspection** — confirm v2 label distribution (no repair needed — Module 1c v2 handled it)
3. **Derived features** — add 6 physics-derived features (same as v1, recomputed on v2 data)
4. **Robust normalisation** — recompute median + IQR scalers on training years only
5. **Class weights** — recompute inverse-frequency weights from updated label counts
6. **Update HDF5** — write updated `mlp_derived` and `mlp_atm` normalised vectors

## Output

| File | Path |
|---|---|
| v2 parquets (final) | `data/processed/viirs_fp_v2/viirs_fp_v2_{YEAR}.parquet` |
| Feature scalers | `data/scalers/feature_scalers_v2.json` |
| Class weights | `data/scalers/class_weights_v2.json` |
| Feature audit figure | `figures/02_feature_audit_v2.png` |

---
## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install pandas numpy h5py pyarrow scipy matplotlib tqdm scikit-learn -q

import os, json, warnings
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

print('All imports OK')

---
## Section 1 — Configuration

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'

# Input: Module 1c v2 parquets
SRF_V2_DIR  = f'{BASE_DIR}/data/processed/viirs_fp_srf_v2'

# Output: final v2 parquets with derived features added
OUT_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_v2'
SCALERS_DIR = f'{BASE_DIR}/data/scalers'
FIGURES_DIR = f'{BASE_DIR}/figures'
ARCHIVE_PATH= f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
SPLIT_DIR   = f'{BASE_DIR}/data/splits'

for d in [OUT_DIR, SCALERS_DIR, FIGURES_DIR, SPLIT_DIR]:
    os.makedirs(d, exist_ok=True)

SCALERS_PATH = f'{SCALERS_DIR}/feature_scalers_v2.json'
WEIGHTS_PATH = f'{SCALERS_DIR}/class_weights_v2.json'

TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR    = 2023
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

# ── Feature column definitions ────────────────────────────────────────────────
ATM_COLS = [
    'era5_t2m','era5_pbl','era5_tcwv','era5_sp',
    'era5_t_1000hPa','era5_t_850hPa','era5_t_700hPa','era5_t_500hPa','era5_t_300hPa',
    'era5_q_1000hPa','era5_q_850hPa','era5_q_700hPa','era5_q_500hPa','era5_q_300hPa',
    'beer_lambert_proxy','atm_instability',
]
SRF_COLS = [
    'lc_forest','lc_shrub','lc_grassland','lc_cropland',
    'lc_urban','lc_bare','lc_water','lc_other',
    'dnb_radiance','dnb_is_persistent',
    'dist_urban_km','dist_powerplant_km','dist_industrial_km',
    'is_urban','is_industrial',
    'sol_zen','is_day',
    'is_prior_burn_scar','burn_scar_age_years',
    'firms_type',
]
DERIVED_COLS = ['aod_3700nm','lifted_index','doy_sin','doy_cos','bt_i4_anom_norm','btd_anom_norm']

ALL_FEATURE_COLS = ATM_COLS + SRF_COLS + DERIVED_COLS

# False-alarm OSM thresholds (same as Module 1c v2)
FA_INDUSTRIAL_KM = 5.0
FA_URBAN_KM      = 2.0
FA_BTD_GATE      = 20.0   # K — below this, industrial/urban pixels are false-alarm

print('Configuration set.')
print(f'  Input  : {SRF_V2_DIR}')
print(f'  Output : {OUT_DIR}')
print(f'  Scalers: {SCALERS_PATH}')

---
## Section 2 — Load v2 parquets and audit features

In [ ]:
all_dfs = {}
for year in ALL_YEARS:
    fp = f'{SRF_V2_DIR}/viirs_fp_srf_v2_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        df['year'] = df['date'].dt.year
        all_dfs[year] = df
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} cols')
    else:
        print(f'  {year}: NOT FOUND — run Module 1c v2 first')

assert all_dfs, 'No v2 parquets found.'
total = sum(len(v) for v in all_dfs.values())
print(f'\nTotal: {total:,} pixels')

# Concatenate for audit
df_all = pd.concat(all_dfs.values(), ignore_index=True)

In [ ]:
# ── Feature audit ─────────────────────────────────────────────────────────────
print(f'Feature audit ({len(ATM_COLS + SRF_COLS)} base features):')
print(f'{"Column":<35} {"Mean|val|":>12} {"NonZero%":>10} {"Status"}')
print('-' * 65)

real_cols, zero_cols = [], []
for col in ATM_COLS + SRF_COLS:
    if col not in df_all.columns:
        print(f'  ✗ {col:<33} MISSING')
        zero_cols.append(col)
        continue
    vals = df_all[col].values.astype(np.float64)
    mean_abs = np.nanmean(np.abs(vals))
    nonzero  = 100.0 * np.isfinite(vals[vals != 0]).sum() / len(vals)
    if mean_abs < 1e-6 and nonzero < 0.1:
        status = 'ZERO'
        zero_cols.append(col)
        icon = '○'
    else:
        status = 'REAL'
        real_cols.append(col)
        icon = '✓'
    print(f'  {icon} {col:<33} {mean_abs:>12.4f} {nonzero:>9.1f}%  {status}')

print(f'\n  {len(real_cols)} real features | {len(zero_cols)} zero-filled')

# Confirm land cover is now real
lc_real = [c for c in real_cols if c.startswith('lc_')]
print(f'  Land cover columns with real data: {lc_real}')

---
## Section 3 — Label inspection

Module 1c v2 already constructed labels using MODIS urban + OSM industrial.
We inspect the distribution here and add a BTD gate on top for extra precision.

In [ ]:
print('Label distribution from Module 1c v2:')
counts = df_all['label'].value_counts().sort_index()
total  = len(df_all)
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = counts.get(lbl, 0)
    bar = '█' * int(n / total * 50)
    print(f'  {lbl} {name:12s}: {n:>9,} ({100*n/total:.2f}%)  {bar}')

# Apply BTD gate refinement: false-alarm pixels that have BTD >= FA_BTD_GATE
# might be genuine fires mislabelled — promote back to wildfire
# This only applies if BTD is available in the parquets
if 'BT_I4' in df_all.columns and 'BT_I5' in df_all.columns:
    df_all['BTD'] = df_all['BT_I4'] - df_all['BT_I5']
    # FA pixels with high BTD: possible fire in industrial zone
    fa_high_btd = (df_all['label'] == 2) & (df_all['BTD'] >= FA_BTD_GATE)
    n_promoted  = fa_high_btd.sum()
    if n_promoted > 0:
        df_all.loc[fa_high_btd, 'label'] = 1  # promote to wildfire
        print(f'\n  BTD gate: promoted {n_promoted:,} high-BTD false-alarms → wildfire')
    else:
        print(f'\n  BTD gate: no false-alarms promoted (all have BTD < {FA_BTD_GATE} K)')
    print(f'  Mean BTD of false-alarm pixels: {df_all[df_all["label"]==2]["BTD"].mean():.1f} K')
    print(f'  Mean BTD of wildfire pixels   : {df_all[df_all["label"]==1]["BTD"].mean():.1f} K')

# Write BTD-corrected labels back to year dicts
if 'BTD' in df_all.columns:
    for year in ALL_YEARS:
        if year in all_dfs:
            mask = df_all['year'] == year
            all_dfs[year]['label'] = df_all.loc[mask, 'label'].values
            if 'BTD' not in all_dfs[year].columns:
                all_dfs[year]['BTD'] = df_all.loc[mask, 'BTD'].values

print('\nFinal label distribution:')
counts2 = df_all['label'].value_counts().sort_index()
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = counts2.get(lbl, 0)
    print(f'  {lbl} {name:12s}: {n:>9,} ({100*n/total:.2f}%)')

---
## Section 4 — Add derived features

Six physics-derived features computed from existing columns.  
Same computation as Module 2 v1, now applied to v2 parquets with real land cover.

In [ ]:
def add_derived_features(df):
    """
    Compute 6 derived physics features.

    aod_3700nm      : AOD at 3.7µm via Ångström exponent (α=1.3)
    lifted_index    : Atmospheric instability proxy (T_500 - T_850) / 50
    doy_sin / doy_cos : Day-of-year circular encoding
    bt_i4_anom_norm : Normalised BT_I4 anomaly above background
    btd_anom_norm   : Normalised BTD anomaly above background
    """
    df = df.copy()
    # AOD at 3.7µm (Ångström exponent scaling from 550nm)
    aod550 = df.get('aod_550', pd.Series(0.1, index=df.index))
    alpha  = 1.3
    df['aod_3700nm'] = aod550 * (550.0 / 3700.0) ** (-alpha)

    # Lifted index
    if 'era5_t_500hPa' in df.columns and 'era5_t_850hPa' in df.columns:
        df['lifted_index'] = (df['era5_t_500hPa'] - df['era5_t_850hPa']) / 50.0
    else:
        df['lifted_index'] = 0.0

    # Day-of-year circular encoding
    doy = df['date'].dt.dayofyear
    df['doy_sin'] = np.sin(2 * np.pi * doy / 365.0)
    df['doy_cos'] = np.cos(2 * np.pi * doy / 365.0)

    # BT anomaly normalised by MAD
    if 'BT_I4' in df.columns and 'BT_I4_bg' in df.columns and 'MAD_I4' in df.columns:
        mad_safe = df['MAD_I4'].clip(lower=1.0)
        df['bt_i4_anom_norm'] = ((df['BT_I4'] - df['BT_I4_bg']) / mad_safe).clip(lower=0)
    else:
        df['bt_i4_anom_norm'] = 0.0

    # BTD anomaly normalised
    if 'BTD' in df.columns and 'BT_I5_bg' in df.columns and 'MAD_I4' in df.columns:
        btd_bg = df.get('BT_I4_bg', 0) - df.get('BT_I5_bg', 0)
        mad_safe = df['MAD_I4'].clip(lower=1.0)
        df['btd_anom_norm'] = ((df['BTD'] - btd_bg) / mad_safe).clip(lower=0)
    else:
        df['btd_anom_norm'] = 0.0

    return df


print('add_derived_features() defined.')
print(f'Adding {len(DERIVED_COLS)} derived features: {DERIVED_COLS}')

In [ ]:
# Apply derived features and save final v2 parquets
print('Applying derived features and saving final v2 parquets...')
all_dfs_final = {}

for year in ALL_YEARS:
    out_path = f'{OUT_DIR}/viirs_fp_v2_{year}.parquet'
    if os.path.exists(out_path):
        df = pd.read_parquet(out_path)
        all_dfs_final[year] = df
        print(f'  {year}: loaded from Drive ({len(df):,} pixels, {len(df.columns)} cols)')
        continue

    if year not in all_dfs:
        print(f'  {year}: missing — skip')
        continue

    df = add_derived_features(all_dfs[year])
    df.to_parquet(out_path, index=False)
    all_dfs_final[year] = df
    print(f'  {year}: {len(df):,} pixels, {len(df.columns)} cols → Drive')

print('\n✓ v2 parquets saved')

---
## Section 5 — Robust scalers on training years

> **Critical:** scalers computed on training years ONLY (2018–2022).  
> The 2023 validation year must never influence normalisation parameters.

In [ ]:
print('Computing robust scalers on training years (2018–2022)...')

train_dfs = [all_dfs_final[y] for y in TRAIN_YEARS if y in all_dfs_final]
df_train  = pd.concat(train_dfs, ignore_index=True)
n_train   = len(df_train)
print(f'  Training pixels: {n_train:,}')

scalers = {}
real_scaled = 0
zero_identity = 0

for col in ALL_FEATURE_COLS:
    if col not in df_train.columns:
        scalers[col] = {'median': 0.0, 'iqr': 1.0, 'status': 'missing'}
        continue

    vals = df_train[col].values.astype(np.float64)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0 or np.all(vals == 0):
        scalers[col] = {'median': 0.0, 'iqr': 1.0, 'status': 'zero'}
        zero_identity += 1
        continue

    median = float(np.median(vals))
    q75, q25 = np.percentile(vals, [75, 25])
    iqr    = float(q75 - q25)
    if iqr < 1e-8:
        iqr = 1.0

    scalers[col] = {'median': median, 'iqr': iqr, 'status': 'scaled'}
    real_scaled += 1

with open(SCALERS_PATH, 'w') as f:
    json.dump(scalers, f, indent=2)

print(f'  {real_scaled} features scaled with real data')
print(f'  {zero_identity} features with identity scaler (zero-filled)')
print(f'  Scalers saved → {SCALERS_PATH}')

# Verify scaling
print('\nScaled feature stats (should be centred near 0):')
check_cols = ['era5_t2m','era5_pbl','era5_tcwv','lc_forest','lc_grassland',
              'dist_industrial_km','doy_sin','bt_i4_anom_norm']
for col in check_cols:
    if col not in df_train.columns or col not in scalers: continue
    s = scalers[col]
    vals = df_train[col].dropna().values.astype(np.float64)
    scaled = (vals - s['median']) / s['iqr']
    print(f'  {col:<30} mean={np.mean(scaled):>7.3f}  std={np.std(scaled):>6.3f}')

---
## Section 6 — Class weights

In [ ]:
print('Computing class weights from training label distribution...')

train_labels = np.concatenate([all_dfs_final[y]['label'].values for y in TRAIN_YEARS
                                if y in all_dfs_final])

n_total = len(train_labels)
n_classes = 3
class_counts = {}
class_weights = {}

print(f'\nTraining set class distribution:')
for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
    n = int((train_labels == lbl).sum())
    class_counts[lbl] = n
    # Inverse frequency: weight = N_total / (N_classes * N_c)
    if n > 0:
        w = n_total / (n_classes * n)
    else:
        w = 1.0  # fallback if class absent
    class_weights[lbl] = round(w, 2)

    bar = '█' * int(w * 1.2) if w < 50 else '█' * 60
    pct = 100 * n / n_total if n_total > 0 else 0
    print(f'  {lbl} {name:12s}: {n:>9,} ({pct:.2f}%)  weight={w:>7.2f}  {bar}')

# Save in both list and dict format for compatibility
weights_list = [class_weights[0], class_weights[1], class_weights[2]]
weights_data = {
    '0': class_weights[0],
    '1': class_weights[1],
    '2': class_weights[2],
    '_list': weights_list,
    '_counts': {str(k): int(v) for k, v in class_counts.items()},
}
with open(WEIGHTS_PATH, 'w') as f:
    json.dump(weights_data, f, indent=2)

print(f'\nClass weights: {weights_list}')
print(f'  Saved → {WEIGHTS_PATH}')
print(f'\nUsage in Module 3:')
print(f'  weights_tensor = torch.tensor({weights_list})')
print(f'  criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(device))')

---
## Section 7 — Update HDF5 archive with normalised feature vectors

In [ ]:
def apply_scalers(df, scalers, feature_cols):
    """Apply saved robust scalers to feature columns. Returns numpy array (N, len(feature_cols))."""
    result = np.zeros((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        if col not in df.columns:
            continue
        vals = df[col].values.astype(np.float64)
        s = scalers.get(col, {'median': 0.0, 'iqr': 1.0})
        scaled = (vals - s['median']) / s['iqr']
        scaled = np.nan_to_num(scaled, nan=0.0).astype(np.float32)
        result[:, i] = scaled
    return result


print(f'Updating HDF5 archive normalised feature vectors...')

with h5py.File(ARCHIVE_PATH, 'a') as hf:
    N = hf['labels'].shape[0]
    print(f'  Archive: {N:,} patches')

    # Ensure mlp_derived dataset exists with correct size
    n_derived = len(DERIVED_COLS)
    if 'mlp_derived' not in hf:
        hf.create_dataset('mlp_derived', shape=(N, n_derived),
                          dtype='float32', chunks=(256, n_derived))
        print(f'  Created mlp_derived dataset: ({N}, {n_derived})')
    else:
        print(f'  mlp_derived exists: shape={hf["mlp_derived"].shape}')

    years_arr = hf['meta/year'][:]

    for year in tqdm(ALL_YEARS, desc='Years'):
        if year not in all_dfs_final:
            continue
        df_year = all_dfs_final[year]

        # Ensure derived feature cols are present
        for col in DERIVED_COLS:
            if col not in df_year.columns:
                df_year[col] = 0.0

        year_idx = np.where(years_arr == year)[0]
        if len(year_idx) == 0:
            print(f'  {year}: no archive entries found')
            continue

        import scipy.spatial
        arch_lats = hf['meta/center_lat'][year_idx]
        arch_lons = hf['meta/center_lon'][year_idx]
        df_pts   = np.column_stack([df_year['latitude'].values, df_year['longitude'].values])
        arch_pts = np.column_stack([arch_lats, arch_lons])

        tree     = scipy.spatial.cKDTree(df_pts)
        _, match = tree.query(arch_pts)

        # Normalised ATM vectors
        atm_scaled = apply_scalers(df_year.iloc[match], scalers, ATM_COLS)

        # Normalised derived vectors
        der_scaled = apply_scalers(df_year.iloc[match], scalers, DERIVED_COLS)

        # Write in sorted order
        sort_order = np.argsort(year_idx)
        sorted_idx = year_idx[sort_order]

        chunk = 10000
        for i in range(0, len(sorted_idx), chunk):
            sl = sorted_idx[i:i+chunk]
            hf['mlp_atm'][sl]     = atm_scaled[sort_order[i:i+chunk]]
            hf['mlp_derived'][sl] = der_scaled[sort_order[i:i+chunk]]

        # Update labels
        label_vals = df_year['label'].values[match][sort_order].astype(np.uint8)
        hf['labels'][sorted_idx] = label_vals

        fa_count = (label_vals == 2).sum()
        print(f'  {year}: {len(year_idx):,} rows updated | false-alarm: {fa_count:,}')

    # Final label check
    final_labels = hf['labels'][:]
    print(f'\nFinal HDF5 label distribution:')
    for lbl, name in [(0,'non-fire'),(1,'wildfire'),(2,'false-alarm')]:
        n = (final_labels == lbl).sum()
        print(f'  {lbl} {name:12s}: {n:>9,} ({100*n/len(final_labels):.2f}%)')

print('\n✓ HDF5 updated')

---
## Section 8 — Verify train/val/test splits

In [ ]:
# Rebuild splits to reflect updated labels
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    all_labels = hf['labels'][:]
    all_years  = hf['meta/year'][:]

val_idx    = np.where(all_years == VAL_YEAR)[0]
train_pool = np.where(all_years != VAL_YEAR)[0]

pool_labels  = all_labels[train_pool]
unique, counts = np.unique(pool_labels, return_counts=True)
can_stratify = len(unique) > 1 and all(c >= 2 for c in counts)

print(f'Train pool label distribution:')
for lbl, cnt in zip(unique, counts):
    name = {0:'non-fire',1:'wildfire',2:'false-alarm'}.get(int(lbl),'?')
    print(f'  {lbl} {name:12s}: {cnt:>9,} ({100*cnt/len(pool_labels):.2f}%)')
print(f'Stratified split: {can_stratify}')

train_idx, test_idx = train_test_split(
    train_pool,
    test_size    = 0.20,
    stratify     = pool_labels if can_stratify else None,
    random_state = 42,
)

np.save(f'{SPLIT_DIR}/train_index.npy', train_idx)
np.save(f'{SPLIT_DIR}/val_index.npy',   val_idx)
np.save(f'{SPLIT_DIR}/test_index.npy',  test_idx)

print(f'\nSplit summary:')
for name, idx in [('Train',train_idx),('Val',val_idx),('Test',test_idx)]:
    lbls  = all_labels[idx]
    lbl_c = {l: int((lbls==l).sum()) for l in [0,1,2]}
    print(f'  {name:5s}: {len(idx):>9,} | '
          f'nf={lbl_c[0]:,} wf={lbl_c[1]:,} fa={lbl_c[2]:,}')

print(f'\n✓ Splits saved → {SPLIT_DIR}')

---
## Section 9 — Feature audit figure

In [ ]:
BG, PANEL, TEXT, SUBTEXT = '#0D1117', '#161B22', '#E6EDF3', '#8B949E'
BORDER, ORANGE, GREEN, RED = '#30363D', '#E85D04', '#16A34A', '#DC2626'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': BORDER, 'axes.labelcolor': TEXT,
    'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'text.color': TEXT, 'grid.color': BORDER,
})

df_sample = pd.concat([all_dfs_final[y] for y in TRAIN_YEARS if y in all_dfs_final],
                       ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(17, 6), facecolor=BG)
fig.patch.set_facecolor(BG)

# ── Panel 1: Label distribution ───────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(PANEL)
lbl_names = ['Non-fire\n(label 0)', 'Wildfire\n(label 1)', 'False-alarm\n(label 2)']
lbl_colors = ['#3A5C8A', ORANGE, '#1565C0']

counts_final = df_sample['label'].value_counts().sort_index()
bar_vals = [counts_final.get(i, 0) for i in range(3)]
bars = ax.bar(lbl_names, bar_vals, color=lbl_colors, width=0.55, alpha=0.85)
for bar, val in zip(bars, bar_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
            f'{val:,}', ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e3:.0f}K'))
ax.set_title('Label distribution (training set)', color=TEXT, fontsize=11)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)

# ── Panel 2: Feature coverage heatmap ─────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(PANEL)
feat_groups = [
    ('ERA5 atm (16)',    ATM_COLS[:4],     True),
    ('ERA5 T levels',   ATM_COLS[4:9],    True),
    ('ERA5 q levels',   ATM_COLS[9:14],   True),
    ('Beer-Lambert/LI', ATM_COLS[14:],    True),
    ('MODIS LC (8)',     SRF_COLS[:8],     None),  # check dynamically
    ('DNB (2)',          SRF_COLS[8:10],   False),
    ('OSM dist (3)',     SRF_COLS[10:13],  True),
    ('OSM flags (2)',    SRF_COLS[13:15],  True),
    ('Solar geo (2)',    SRF_COLS[15:17],  True),
    ('Burn scars (2)',   SRF_COLS[17:19],  False),
    ('firms_type (1)',   SRF_COLS[19:],    False),
    ('Derived (6)',      DERIVED_COLS,     True),
]

group_names = [g[0] for g in feat_groups]
y_pos = np.arange(len(group_names))
for i, (name, cols, is_real) in enumerate(feat_groups):
    # Dynamically check if MODIS LC is real
    if is_real is None:
        sample_col = cols[0] if cols else None
        if sample_col and sample_col in df_sample.columns:
            nonzero_pct = (df_sample[sample_col] != 0).mean() * 100
            is_real = nonzero_pct > 1.0
        else:
            is_real = False
    color = GREEN if is_real else RED
    ax2.barh(i, 1, color=color, alpha=0.75, height=0.75)
    status = 'Real data' if is_real else 'Zero-filled'
    ax2.text(0.5, i, f'{name}  —  {status}', ha='center', va='center',
             fontsize=8.5, color='white' if is_real else '#FFAAAA', fontweight='bold')

ax2.set_yticks(y_pos); ax2.set_yticklabels([])
ax2.set_xticks([])
ax2.invert_yaxis()
ax2.set_title('Feature coverage\n(green=real, red=zero-filled)', color=TEXT, fontsize=11)
handles = [mpatches.Patch(color=GREEN, label='Real data'), mpatches.Patch(color=RED, label='Zero-filled')]
ax2.legend(handles=handles, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8,
           loc='lower right')
ax2.spines[['top','right','bottom','left']].set_visible(False)

# ── Panel 3: Class weights ──────────────────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor(PANEL)
cls_names  = ['Non-fire\n(label 0)', 'Wildfire\n(label 1)', 'False-alarm\n(label 2)']
w_vals     = [class_weights[0], class_weights[1], class_weights[2]]
w_colors   = ['#3A5C8A', ORANGE, '#1565C0']
bars3 = ax3.bar(cls_names, w_vals, color=w_colors, width=0.55, alpha=0.85)
for bar, val in zip(bars3, w_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
             f'×{val:.1f}', ha='center', fontsize=10, color=TEXT, fontweight='bold')
ax3.set_ylabel('Loss weight multiplier', color=SUBTEXT)
ax3.set_title('Class weights for weighted CE loss\n(higher = penalised more for missing)', color=TEXT, fontsize=11)
ax3.spines[['top','right']].set_visible(False)
ax3.grid(axis='y', alpha=0.2)
ax3.text(0.5, 0.97,
         'These weights penalise the model more for missing\nrare classes (non-fire, false-alarm)',
         transform=ax3.transAxes, ha='center', va='top', fontsize=8,
         color=SUBTEXT, style='italic')

fig.suptitle('FireSight-IR  |  Module 2 v2 — Feature Engineering Audit',
             color=TEXT, fontsize=13, y=1.02)
plt.tight_layout()

out = f'{FIGURES_DIR}/02_feature_audit_v2.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 10 — Final HDF5 structure verification

In [ ]:
print('=== Final HDF5 Archive Structure ===')
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    def walk(h5, prefix=''):
        for k in h5.keys():
            item = h5[k]
            path = f'{prefix}/{k}'
            if isinstance(item, h5py.Dataset):
                print(f'  {path:<35} shape={str(item.shape):<28} dtype={item.dtype}')
            else:
                walk(item, path)
    walk(hf)

    print('\n=== Normalised ATM feature stats (should be ~N(0,1)) ===')
    atm_sample = hf['mlp_atm'][:10000]
    for i, col in enumerate(ATM_COLS[:4]):
        v = atm_sample[:, i]
        print(f'  {col:<28} mean={np.mean(v):.3f}  std={np.std(v):.3f}')

    print('\n=== Derived feature stats ===')
    if 'mlp_derived' in hf:
        der_sample = hf['mlp_derived'][:10000]
        for i, col in enumerate(DERIVED_COLS):
            v = der_sample[:, i]
            print(f'  {col:<28} mean={np.mean(v):.3f}  std={np.std(v):.3f}')

print('\n=== Summary ===')
print(f'  Scalers path : {SCALERS_PATH}')
print(f'  Weights path : {WEIGHTS_PATH}')
print(f'  Class weights: {weights_list}')
print(f'\n  Next: Delete data/cache/*.npy, then run Module 3a (fast) to rebuild cache and train.')